In [20]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "allens-civic-ops-project"  # your actual project ID
DATASET_ID = "civic_ops"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
print("Connected to:", client.project)

Connected to: allens-civic-ops-project


In [21]:
# Cell: Create the destination table in BigQuery (run once)
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)
table_id = f"{PROJECT_ID}.{DATASET_ID}.raw_311_requests"

# We'll let the first load auto-detect schema, then reuse the table for appends
print("Target table:", table_id)

Target table: allens-civic-ops-project.civic_ops.raw_311_requests


In [ ]:
# Cell 2: Pull NYC 311 data from the live Socrata API
import requests
import pandas as pd

BASE_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
LIMIT = 50000
offset = 0
where_clause = "created_date >= '2026-02-01T00:00:00'"
table_id = f"{PROJECT_ID}.{DATASET_ID}.raw_311_requests"

job_config = bigquery.LoadJobConfig(
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)

total_loaded = 0
batch_num = 0

while True:
    params = {
        "$limit": LIMIT,
        "$offset": offset,
        "$where": where_clause,
        "$order": "created_date"
    }
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    batch = resp.json()
    if not batch:
        break

    df_batch = pd.DataFrame(batch)
    load_job = client.load_table_from_dataframe(df_batch, table_id, job_config=job_config)
    load_job.result()  # wait for load to finish

    batch_num += 1
    total_loaded += len(df_batch)
    offset += LIMIT
    print(f"Batch {batch_num}: loaded {len(df_batch)} rows (total: {total_loaded})")

    del df_batch  # free memory immediately

print("Done. Total rows loaded:", total_loaded)

Batch 1: loaded 50000 rows (total: 50000)
Batch 2: loaded 50000 rows (total: 100000)
Batch 3: loaded 50000 rows (total: 150000)
Batch 4: loaded 50000 rows (total: 200000)
Batch 5: loaded 50000 rows (total: 250000)
Batch 6: loaded 50000 rows (total: 300000)
Batch 7: loaded 50000 rows (total: 350000)
Batch 8: loaded 50000 rows (total: 400000)
Batch 9: loaded 50000 rows (total: 450000)
Batch 10: loaded 50000 rows (total: 500000)
Batch 11: loaded 50000 rows (total: 550000)
Batch 12: loaded 50000 rows (total: 600000)
Batch 13: loaded 50000 rows (total: 650000)
Batch 14: loaded 50000 rows (total: 700000)
Batch 15: loaded 50000 rows (total: 750000)
Batch 16: loaded 50000 rows (total: 800000)
Batch 17: loaded 50000 rows (total: 850000)
Batch 18: loaded 50000 rows (total: 900000)
Batch 19: loaded 50000 rows (total: 950000)
Batch 20: loaded 50000 rows (total: 1000000)
Batch 21: loaded 50000 rows (total: 1050000)
Batch 22: loaded 50000 rows (total: 1100000)
Batch 23: loaded 50000 rows (total: 115

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
LIMIT = 50000
offset = 0
where_clause = "created_date >= '2026-02-01T00:00:00'"
table_id = f"{PROJECT_ID}.{DATASET_ID}.raw_311_requests"

total_loaded = 0
batch_num = 0

while True:
    params = {
        "$limit": LIMIT,
        "$offset": offset,
        "$where": where_clause,
        "$order": "created_date"
    }
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    batch = resp.json()
    if not batch:
        break

    df_batch = pd.DataFrame(batch)

    # Truncate only on the very first batch; append after that
    write_mode = "WRITE_TRUNCATE" if batch_num == 0 else "WRITE_APPEND"
    job_config = bigquery.LoadJobConfig(autodetect=True, write_disposition=write_mode)

    load_job = client.load_table_from_dataframe(df_batch, table_id, job_config=job_config)
    load_job.result()

    batch_num += 1
    total_loaded += len(df_batch)
    offset += LIMIT
    print(f"Batch {batch_num}: loaded {len(df_batch)} rows (total: {total_loaded}, mode: {write_mode})")

    del df_batch

print("Done. Total rows loaded:", total_loaded)

Batch 1: loaded 50000 rows (total: 50000, mode: WRITE_TRUNCATE)
Batch 2: loaded 50000 rows (total: 100000, mode: WRITE_APPEND)
Batch 3: loaded 50000 rows (total: 150000, mode: WRITE_APPEND)
Batch 4: loaded 50000 rows (total: 200000, mode: WRITE_APPEND)
Batch 5: loaded 50000 rows (total: 250000, mode: WRITE_APPEND)
Batch 6: loaded 50000 rows (total: 300000, mode: WRITE_APPEND)
Batch 7: loaded 50000 rows (total: 350000, mode: WRITE_APPEND)
Batch 8: loaded 50000 rows (total: 400000, mode: WRITE_APPEND)
Batch 9: loaded 50000 rows (total: 450000, mode: WRITE_APPEND)
Batch 10: loaded 50000 rows (total: 500000, mode: WRITE_APPEND)
Batch 11: loaded 50000 rows (total: 550000, mode: WRITE_APPEND)
Batch 12: loaded 50000 rows (total: 600000, mode: WRITE_APPEND)
Batch 13: loaded 50000 rows (total: 650000, mode: WRITE_APPEND)
Batch 14: loaded 50000 rows (total: 700000, mode: WRITE_APPEND)
Batch 15: loaded 50000 rows (total: 750000, mode: WRITE_APPEND)
Batch 16: loaded 50000 rows (total: 800000, mode

In [1]:
!pip install -q langchain langchain-google-genai google-cloud-bigquery langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 945.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73

In [24]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GEMINI_API_KEY,
    temperature=0
)

# Quick sanity check
response = llm.invoke("Say hello in one sentence.")
print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Hello and welcome!', 'extras': {'signature': 'El4KXAFpFH0TsvasOZeD/Y/GkAwaxowFDYn7sM6IJYXI7cCkZNSKJ0QmQJ/nsoMuAGjTzSC1OsUqOHKMKvEvpWkM7u/7k25JIzKWdKwdwz4+ZnnqOPsUA8uhWYRGAId3'}}]


In [7]:
schema_context = """
You are querying a BigQuery table called `allens-civic-ops-project.civic_ops.raw_311_requests`.
It contains NYC 311 service request data with these key columns:
- created_date (TIMESTAMP): when the request was filed
- complaint_type (STRING): category of complaint (e.g., "Illegal Parking", "Noise - Residential")
- borough (STRING): one of BROOKLYN, QUEENS, BRONX, MANHATTAN, STATEN ISLAND, or Unspecified
- agency (STRING): responding city agency
- status (STRING): request status
- closed_date (TIMESTAMP): when the request was resolved, if closed

The data covers roughly February 2026 through August 2026.
"""

In [26]:
from langchain_core.prompts import ChatPromptTemplate

sql_prompt = ChatPromptTemplate.from_messages([
    ("system", schema_context + """
Given a user's question, write a single BigQuery SQL query that answers it.
Rules:
- Only output the SQL query, no explanation, no markdown formatting, no backticks around the query.
- Always use the full table path: `allens-civic-ops-project.civic_ops.raw_311_requests`
- Use standard BigQuery SQL syntax.
"""),
    ("human", "{question}")
])

def generate_sql(question):
    chain = sql_prompt | llm
    result = chain.invoke({"question": question})

    # Handle both string and list content formats (model-dependent)
    content = result.content
    if isinstance(content, list):
        content = "".join(
            part if isinstance(part, str) else part.get("text", "")
            for part in content
        )

    sql = content.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql

# Test it
test_sql = generate_sql("How many noise complaints were there in Brooklyn?")
print(test_sql)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


SELECT COUNT(*) FROM `allens-civic-ops-project.civic_ops.raw_311_requests` WHERE borough = 'BROOKLYN' AND complaint_type LIKE '%Noise%'


In [30]:
def ask_nyc311(question):
    # Step 1: Generate SQL from the question
    sql = generate_sql(question)
    print(f"Generated SQL:\n{sql}\n")

    # Step 2: Execute it against BigQuery
    try:
        result_df = client.query(sql).to_dataframe()
    except Exception as e:
        return f"I couldn't run that query. Error: {e}"

    # Step 3: Turn the result into a plain-English answer
    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", "You answer questions about NYC 311 data based on query results. Be concise and specific, referencing actual numbers from the data."),
        ("human", f"Question: {question}\n\nQuery result:\n{result_df.to_string()}\n\nAnswer the question in 1-2 sentences based on this result.")
    ])
    chain = answer_prompt | llm
    answer = chain.invoke({})

    return extract_text(answer.content)

# Test it end-to-end
print(ask_nyc311("How many noise complaints were there in Brooklyn?"))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Generated SQL:
SELECT count(*) FROM `allens-civic-ops-project.civic_ops.raw_311_requests` WHERE borough = 'BROOKLYN' AND complaint_type LIKE '%Noise%'



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


There were 111,109 noise complaints in Brooklyn.


In [29]:
def extract_text(content):
    """Normalize Gemini response content to a plain string, regardless of model format."""
    if isinstance(content, list):
        return "".join(
            part if isinstance(part, str) else part.get("text", "")
            for part in content
        )
    return content

In [15]:
import time

def ask_nyc311(question, max_retries=3):
    sql = generate_sql(question)
    print(f"Generated SQL:\n{sql}\n")

    try:
        result_df = client.query(sql).to_dataframe()
    except Exception as e:
        return f"I couldn't run that query. Error: {e}"

    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", "You answer questions about NYC 311 data based on query results. Be concise and specific, referencing actual numbers from the data."),
        ("human", f"Question: {question}\n\nQuery result:\n{result_df.to_string()}\n\nAnswer the question in 1-2 sentences based on this result.")
    ])
    chain = answer_prompt | llm

    for attempt in range(max_retries):
        try:
            answer = chain.invoke({})
            return answer.content
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                wait_time = 15 * (attempt + 1)
                print(f"Rate limited, waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                return f"Error generating answer: {e}"
    return "Failed after retries due to rate limiting."

In [16]:
for q in test_questions:
    print(f"Q: {q}")
    print(ask_nyc311(q))
    print("-" * 60)
    time.sleep(12)

Q: How many total requests were there in February 2026?
Generated SQL:
SELECT
    COUNT(*)
FROM
    `allens-civic-ops-project.civic_ops.raw_311_requests`
WHERE
    created_date >= '2026-02-01'
    AND created_date < '2026-03-01'

There were a total of 334,691 requests in February 2026.
------------------------------------------------------------
Q: What is the most common complaint type in Manhattan?
Generated SQL:
SELECT
    complaint_type
FROM
    `allens-civic-ops-project.civic_ops.raw_311_requests`
WHERE
    borough = 'MANHATTAN'
GROUP BY
    complaint_type
ORDER BY
    COUNT(*) DESC
LIMIT 1

The most common complaint type in Manhattan is Illegal Parking.
------------------------------------------------------------
Q: Which borough had the most heating complaints?
Generated SQL:
SELECT
    borough
FROM
    `allens-civic-ops-project.civic_ops.raw_311_requests`
WHERE
    complaint_type = 'HEAT/HOT WATER'
GROUP BY
    borough
ORDER BY
    COUNT(*) DESC
LIMIT 1

The Bronx had the most 

In [17]:
# Eval set: question + expected value(s) that should appear in the answer
eval_set = [
    {
        "question": "How many requests were filed on 2026-02-24?",
        "expected_value": "22,806",
        "notes": "Known anomaly, cross-validated against anomaly detector"
    },
    {
        "question": "How many total requests were there in February 2026?",
        "expected_value": "334,691",
        "notes": "Validated in prior test run"
    },
    {
        "question": "What is the most common complaint type in Manhattan?",
        "expected_value": "Illegal Parking",
        "notes": "Validated in prior test run"
    },
    {
        "question": "Which borough had the most heating complaints?",
        "expected_value": "Bronx",
        "notes": "Validated in prior test run"
    },
    {
        "question": "How many noise complaints were there in Brooklyn?",
        "expected_value": "111,109",
        "notes": "Validated in prior test run"
    },
]

In [31]:
import time

def run_eval(eval_set, delay=12):
    results = []
    for i, case in enumerate(eval_set):
        print(f"[{i+1}/{len(eval_set)}] Q: {case['question']}")
        answer = ask_nyc311(case['question'])
        passed = case['expected_value'].lower() in answer.lower()

        results.append({
            "question": case['question'],
            "expected": case['expected_value'],
            "answer": answer,
            "passed": passed,
            "notes": case.get('notes', '')
        })

        status = "PASS" if passed else "FAIL"
        print(f"  {status} | Expected: {case['expected_value']}")
        print(f"  Answer: {answer}")
        print("-" * 60)

        if i < len(eval_set) - 1:
            time.sleep(delay)

    return results

eval_results = run_eval(eval_set)

# Summary
passed_count = sum(r['passed'] for r in eval_results)
print(f"\n=== EVAL SUMMARY: {passed_count}/{len(eval_results)} passed ===")

[1/5] Q: How many requests were filed on 2026-02-24?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Generated SQL:
SELECT count(*) FROM allens-civic-ops-project.civic_ops.raw_311_requests WHERE date(created_date) = '2026-02-24'



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  PASS | Expected: 22,806
  Answer: There were 22,806 requests filed on 2026-02-24.
------------------------------------------------------------
[2/5] Q: How many total requests were there in February 2026?


...

  FAIL | Expected: Bronx
  Answer: Based on the query results, there is no data available to determine which borough had the most heating complaints, as the returned dataset is empty.
------------------------------------------------------------
[5/5] Q: How many noise complaints were there in Brooklyn?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Generated SQL:
SELECT count(*) FROM allens-civic-ops-project.civic_ops.raw_311_requests WHERE borough = 'BROOKLYN' AND complaint_type LIKE '%Noise%'



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  PASS | Expected: 111,109
  Answer: There were 111,109 noise complaints in Brooklyn based on the 311 data.
------------------------------------------------------------

=== EVAL SUMMARY: 4/5 passed ===


In [ ]:
# Pull daily volume, excluding the last (likely partial) day
import pandas as pd
query = """
SELECT DATE(created_date) AS day, COUNT(*) AS request_count
FROM `allens-civic-ops-project.civic_ops.raw_311_requests`
WHERE DATE(created_date) < (SELECT MAX(DATE(created_date)) FROM `allens-civic-ops-project.civic_ops.raw_311_requests`)
GROUP BY day
ORDER BY day
"""

daily_df = client.query(query).to_dataframe()
daily_df['day'] = pd.to_datetime(daily_df['day'])
daily_df = daily_df.set_index('day')
print(daily_df.shape)
daily_df.tail(10)

(185, 1)


,request_count
day,
2026-07-26,11563
2026-07-27,11252
2026-07-28,10468
2026-07-29,10410
2026-07-30,9555
2026-07-31,10644
2026-08-01,9987
2026-08-02,9834
2026-08-03,11170


In [ ]:
# Rolling 7-day average + z-score anomaly detection
window = 7

daily_df['rolling_mean'] = daily_df['request_count'].rolling(window=window, center=False).mean()
daily_df['rolling_std'] = daily_df['request_count'].rolling(window=window, center=False).std()

daily_df['z_score'] = (daily_df['request_count'] - daily_df['rolling_mean']) / daily_df['rolling_std']

# Flag anomalies: |z| > 2.5 is a common threshold (roughly 99% confidence)
threshold = 2.5
daily_df['is_anomaly'] = daily_df['z_score'].abs() > threshold

anomalies = daily_df[daily_df['is_anomaly']]
print(f"Found {len(anomalies)} anomalous days out of {len(daily_df)}")
anomalies[['request_count', 'rolling_mean', 'z_score']]

# Diagnose: look at top z-scores regardless of threshold
daily_df_sorted = daily_df.dropna(subset=['z_score']).sort_values('z_score', ascending=False)
print("Top 10 highest z-scores:")
daily_df_sorted[['request_count', 'rolling_mean', 'rolling_std', 'z_score']].head(10)

# Fix: exclude the current day from its own rolling baseline
window = 7

daily_df['rolling_mean'] = daily_df['request_count'].shift(1).rolling(window=window).mean()
daily_df['rolling_std'] = daily_df['request_count'].shift(1).rolling(window=window).std()

daily_df['z_score'] = (daily_df['request_count'] - daily_df['rolling_mean']) / daily_df['rolling_std']

threshold = 2.5
daily_df['is_anomaly'] = daily_df['z_score'].abs() > threshold

anomalies = daily_df[daily_df['is_anomaly']]
print(f"Found {len(anomalies)} anomalous days out of {len(daily_df)}")
anomalies[['request_count', 'rolling_mean', 'rolling_std', 'z_score']]

Found 0 anomalous days out of 185
Top 10 highest z-scores:
Found 10 anomalous days out of 185


,request_count,rolling_mean,rolling_std,z_score
day,,,,
2026-02-23,15424,9845.714286,674.691958,8.2679
2026-02-24,22806,10737.428571,2154.140576,5.602499
2026-03-09,14146,11007.000000,750.774489,4.181016
2026-04-13,11118,9844.428571,469.981155,2.709835
2026-04-14,11963,9936.857143,644.836524,3.142103
2026-05-19,12339,11030.000000,432.947264,3.023463
2026-05-23,7972,11670.285714,779.455732,-4.744703
2026-06-07,12559,11406.571429,425.646120,2.707481
2026-07-02,14139,11104.714286,649.333688,4.672922


In [ ]:
# Check day-of-week and nearby context for the flagged dips
check_dates = ['2026-05-23', '2026-07-11']

for date in check_dates:
    row = daily_df.loc[date]
    day_of_week = pd.Timestamp(date).day_name()
    print(f"{date} ({day_of_week}): count={row['request_count']}, "
          f"rolling_mean={row['rolling_mean']:.0f}, z={row['z_score']:.2f}")

# Also show the surrounding week for each, to spot a pattern
print("\n--- Context around 2026-05-23 ---")
print(daily_df.loc['2026-05-18':'2026-05-27', ['request_count', 'z_score']])

print("\n--- Context around 2026-07-11 ---")
print(daily_df.loc['2026-07-06':'2026-07-15', ['request_count', 'z_score']])

2026-05-23 (Saturday): count=7972, rolling_mean=11670, z=-4.74
2026-07-11 (Saturday): count=9179, rolling_mean=11673, z=-3.04

--- Context around 2026-05-23 ---
            request_count   z_score
day                                
2026-05-18          11480   1.32529
2026-05-19          12339  3.023463
2026-05-20          12733  2.306797
2026-05-21          12095  0.721768
2026-05-22          10568 -1.750975
2026-05-23           7972 -4.744703
2026-05-24           8254 -1.868321
2026-05-25          10072 -0.362008
2026-05-26          12512  0.999789
2026-05-27          11486  0.450814

--- Context around 2026-07-11 ---
            request_count   z_score
day                                
2026-07-06          11473 -1.026872
2026-07-07          11607 -0.779779
2026-07-08          11542 -0.921997
2026-07-09          10908 -1.526526
2026-07-10          10753 -1.462618
2026-07-11           9179 -3.041249
2026-07-12          10383 -0.733398
2026-07-13          11613  0.902218
2026-07-14  

In [ ]:
n_weeks = 4
min_history = 3  # require at least 3 prior same-weekdays before scoring

dow_mean = []
dow_std = []

for idx in daily_df.index:
    dow = daily_df.loc[idx, 'day_of_week']
    same_dow_history = daily_df[(daily_df['day_of_week'] == dow) & (daily_df.index < idx)]
    recent = same_dow_history['request_count'].tail(n_weeks)

    if len(recent) >= min_history:
        dow_mean.append(recent.mean())
        dow_std.append(recent.std())
    else:
        dow_mean.append(None)
        dow_std.append(None)

daily_df['dow_rolling_mean'] = dow_mean
daily_df['dow_rolling_std'] = dow_std
daily_df['dow_z_score'] = (daily_df['request_count'] - daily_df['dow_rolling_mean']) / daily_df['dow_rolling_std']

threshold = 2.5
daily_df['is_dow_anomaly'] = daily_df['dow_z_score'].abs() > threshold

dow_anomalies = daily_df[daily_df['is_dow_anomaly']].dropna(subset=['dow_z_score'])
print(f"Found {len(dow_anomalies)} anomalous days (day-of-week-aware) out of {len(daily_df)}")
dow_anomalies[['day_of_week', 'request_count', 'dow_rolling_mean', 'dow_z_score']]

Found 23 anomalous days (day-of-week-aware) out of 185


,day_of_week,request_count,dow_rolling_mean,dow_z_score
day,,,,
2026-02-24,Tuesday,22806,11979.666667,6.817565
2026-02-25,Wednesday,13703,11118.333333,2.6674
2026-02-26,Thursday,12186,11096.333333,2.500121
2026-03-06,Friday,12335,10711.000000,5.81304
2026-04-02,Thursday,9650,11199.750000,-4.975377
2026-04-04,Saturday,8816,9773.000000,-5.947044
2026-04-05,Sunday,7800,10046.750000,-3.164528
2026-04-07,Tuesday,10415,12043.250000,-3.618289
2026-04-08,Wednesday,9966,11356.750000,-2.86503
